# Synthetic camera RAW 50 MP Colab

Incremental implementation of PRD_v3. Output will be a **synthetic sensor reconstruction of a hypothetical
exposure**, never a genuine camera capture. The early RAW gate exports tiny test DNGs only; full reconstruction and TIFF export remain pending.
Metadata must carry D/I/F/M provenance; DNG Make must be `Synthetic`. No Sony MakerNotes,
realistic Sony serials, date inference or GPS inference are permitted.

Run in Colab only. Configure the panel before running Input audit. `FAST_DEV_RUN` reduces planned
sensor dimensions; SR reconstructs on disk using the planned sensor geometry.
Learned weights require explicit licence acceptance and a Colab runtime. Compute stages benchmark initial tiles,
checkpoint to host RAM/disk and release GPU tensors between stages.


## Environment + reproducibility

In [ ]:
# Colab setup: replace this branch fallback with a reviewed 40-character commit SHA
# for a reproducible release. REPO_REF is the commit SHA placeholder until then.
import re
import subprocess
import sys

REPO_REF = "fm/exp50mp-v3-synthetic-raw"
if REPO_REF != "fm/exp50mp-v3-synthetic-raw" and not re.fullmatch(r"[0-9a-fA-F]{40}", REPO_REF):
    raise ValueError("Use the documented branch fallback or a full commit SHA")
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        f"git+https://github.com/RDC28/experiment_50MP.git@{REPO_REF}",
    ]
)
# This repository distribution installs the top-level synthraw package.

### Configuration

Every field comes from `ExperimentConfig`. Edit controls, then run Input audit to validate and freeze
settings. PATH uses a runtime path. DRIVE mounts Drive and uses INPUT_PATH inside that mount.
CALIBRATION_ASSETS accepts a JSON list of runtime paths. Leave INPUT_PATH empty for UPLOAD.
If ipywidgets is unavailable, edit `CONFIG_VALUES` in this cell and rerun it. Changing configuration
requires rerunning Input audit and subsequent cells to start a fresh run.


In [ ]:
import json
from dataclasses import fields
from typing import Literal, get_args, get_origin, get_type_hints

from IPython.display import display

from synthraw.config import ExperimentConfig

CONFIG_VALUES = ExperimentConfig().to_dict()
controls = {}
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

if widgets is None:
    print("ipywidgets unavailable: edit CONFIG_VALUES above, then run Input audit.")
    print(json.dumps(CONFIG_VALUES, indent=2))
else:
    hints = get_type_hints(ExperimentConfig)
    for field in fields(ExperimentConfig):
        name, value, hint = field.name, CONFIG_VALUES[field.name], hints[field.name]
        options = dict(description=name, style={"description_width": "260px"}, layout=widgets.Layout(width="100%"))
        if get_origin(hint) is Literal:
            control = widgets.Dropdown(options=get_args(hint), value=value, **options)
        elif hint is bool:
            control = widgets.Checkbox(value=value, **options)
        elif hint is int:
            # Text avoids imposing widget integer limits on NumPy seed values.
            control = widgets.Text(value=str(value), **options)
        elif hint is float:
            control = widgets.FloatText(value=value, **options)
        else:
            text = json.dumps(value) if name == "CALIBRATION_ASSETS" else (value or "")
            control = widgets.Text(value=text, **options)
        controls[name] = control
    display(widgets.VBox(list(controls.values())))


def read_configuration():
    """Validate current controls, including dependent choices, before any input I/O."""
    if not controls:
        return ExperimentConfig.from_dict(CONFIG_VALUES)
    values = {name: control.value for name, control in controls.items()}
    for field in fields(ExperimentConfig):
        if get_type_hints(ExperimentConfig)[field.name] is int:
            values[field.name] = int(values[field.name])
    values["INPUT_PATH"] = values["INPUT_PATH"] or None
    values["CALIBRATION_ASSETS"] = json.loads(values["CALIBRATION_ASSETS"])
    return ExperimentConfig.from_dict(values)

In [ ]:
from synthraw.environment import EnvironmentInspector

environment = EnvironmentInspector().inspect()
EnvironmentInspector.print_report(environment)

### Mandatory tiny RAW gate

Run this before Input audit or any reconstruction. Eight 64 x 48 Bayer fixtures test all CFA
phases at both ADC depths, including the experimental virtual 16-bit profile. Each DNG must
reopen with identical pixels and matching metadata through independent LibRaw and develop to
finite 16-bit RGB. Any failure stops execution; fix the runtime before continuing. Temporary
DNGs are removed on success or failure. These fixtures are synthetic test data, not the input image.

In [ ]:
# Mandatory RAW gate: only 64 x 48 fixtures, before any reconstruction.
from dataclasses import replace
from tempfile import TemporaryDirectory

import numpy as np

from synthraw.dng_metadata import CFA_PATTERNS, DNGMetadataBuilder
from synthraw.dng_writer import DNGWriter
from synthraw.libraw_validator import LibRawValidator

RAW_GATE_PASSED = False


def run_raw_gate():
    """Fail immediately if independent LibRaw decoding or development fails."""
    from pathlib import Path

    from synthraw.geometry import TargetGeometryPlanner

    tiny_geometry = replace(
        TargetGeometryPlanner().plan(64, 48),
        width=64,
        height=48,
        active_area=(0, 0, 48, 64),
        default_crop_origin=(0, 0),
        default_crop_size=(64, 48),
    )
    with TemporaryDirectory(prefix="synthraw_gate_") as directory:
        for bits in (14, 16):
            gate_config = ExperimentConfig(ADC_BITS=bits)
            for phase in CFA_PATTERNS:
                records = DNGMetadataBuilder().build(
                    tiny_geometry,
                    gate_config,
                    cfa_phase=phase,
                    as_shot_neutral=(0.5, 1, 0.75),
                    color_matrix2=((0.8, 0.1, 0.05), (0.05, 1, 0.1), (0.02, 0.1, 0.9)),
                )
                mosaic = np.arange(48 * 64, dtype=np.uint16).reshape(48, 64) + 512
                path = DNGWriter().write(Path(directory) / f"{phase}_{bits}.dng", mosaic, records, gate_config)
                LibRawValidator().validate(path, mosaic, records, develop=True)


run_raw_gate()
RAW_GATE_PASSED = True
print("RAW gate passed: all four Bayer phases at 14 and virtual 16 bits; LibRaw pixels, metadata and RGB verified.")

## Input audit

In [ ]:
if not globals().get("RAW_GATE_PASSED", False):
    raise RuntimeError("Run the mandatory tiny RAW gate before Input audit")

import shutil
from pathlib import Path

from synthraw.input_audit import InputInspector
from synthraw.run_layout import RandomStreams, RunLayout

config = read_configuration()
if config.INPUT_MODE == "UPLOAD":
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one JPEG or PNG")
    source_path = Path(next(iter(uploaded)))
    del uploaded  # Colab already saved the bytes to source_path.
else:
    if config.INPUT_MODE == "DRIVE":
        from google.colab import drive

        drive.mount(config.DRIVE_MOUNT_PATH)
    source_path = Path(config.INPUT_PATH).expanduser()
    if config.INPUT_MODE == "DRIVE" and not source_path.resolve().is_relative_to(
        Path(config.DRIVE_MOUNT_PATH).resolve()
    ):
        raise ValueError("DRIVE INPUT_PATH must be inside DRIVE_MOUNT_PATH")

audit = InputInspector().inspect(source_path)
run = RunLayout.create("/content/synthraw_runs")
streams = RandomStreams.from_seeds(config.SCENE_SEED, config.SENSOR_NOISE_SEED, config.SENSOR_BODY_SEED)
shutil.copy2(source_path, run.directory("source") / source_path.name)
(run.directory("metadata") / "config.json").write_text(json.dumps(config.to_dict(), indent=2), encoding="utf-8")
(run.directory("metadata") / "input_audit.json").write_text(json.dumps(audit.to_dict(), indent=2), encoding="utf-8")
print(json.dumps(audit.to_dict(), indent=2))
print(f"Run directory: {run.root}")

## Profile selection

In [ ]:
print(f"Target profile: {config.TARGET_PROFILE}")
print(f"Profile source: {config.CAMERA_PROFILE_SOURCE}; spectral mode: {config.SPECTRAL_MODE}")
# The camera section below validates the selected profile and writes its provenance.

## Target geometry

In [ ]:
from dataclasses import asdict

from synthraw.geometry import TargetGeometryPlanner

geometry = TargetGeometryPlanner().plan(audit.width, audit.height, config)
geometry_json = json.dumps(asdict(geometry), indent=2)
(run.directory("metadata") / "geometry_plan.json").write_text(geometry_json, encoding="utf-8")
print(geometry_json)
print("Geometry and memory estimates only; no full sensor arrays allocated.")

## Source diagnostics

In [ ]:
print(f"Source: {audit.resolution_class}, aspect {audit.aspect_ratio:.6f}")
print(f"Near-black RGB fractions: {audit.near_black_fractions}")
print(f"Near-white RGB fractions: {audit.near_white_fractions}")
# Reconstruction milestone: add spatial detail, ringing and compression diagnostics.

## SR reconstruction

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

from PIL import Image

from synthraw.provenance import ProvenanceWriter
from synthraw.super_resolution import SuperResolutionEngine

# Set True only after reviewing and accepting the selected checkpoint licence.
ACCEPT_SR_MODEL_LICENCE = False

with Image.open(source_path) as image:
    # Match InputInspector's native pixel orientation and RGB decoding.
    input_rgb = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
if input_rgb.shape != (audit.height, audit.width, 3):
    raise ValueError("Decoded source dimensions differ from the input audit")
input_path = run.directory("intermediates") / "01_input_rgb.npy"
with input_path.open("xb") as stream:
    np.save(stream, input_rgb, allow_pickle=False)
del input_rgb
input_rgb = np.load(input_path, mmap_mode="r")

sr_result = SuperResolutionEngine().run(
    input_rgb,
    geometry,
    config,
    environment,
    run.root,
    accept_licence=ACCEPT_SR_MODEL_LICENCE,
)
sr_rgb = np.load(sr_result.canvas.path, mmap_mode="r")
# Paths are serialized explicitly; image products remain disk-backed.
sr_report = asdict(sr_result)
(run.directory("metadata") / "sr_reconstruction.json").write_text(
    json.dumps(
        sr_report, indent=2, default=lambda value: str(value) if isinstance(value, Path) else value, allow_nan=False
    )
    + "\n",
    encoding="utf-8",
)
ProvenanceWriter().write([sr_result.canvas.provenance], run.directory("metadata") / "sr_provenance.json")
print(f"SR backend: {sr_result.tiled.backend}; fallback reasons: {sr_result.tiled.fallback_reasons}")
print(
    f"Tile benchmark: {sr_result.tiled.benchmark_tile_count} tiles; estimated tile seconds: "
    f"{sr_result.tiled.estimated_seconds:.2f}"
)
print(f"SR canvas: {sr_result.canvas.path}; source fidelity: {sr_result.source_fidelity}")

## HDR / scene-linear reconstruction

In [ ]:
import json
import tempfile

import matplotlib.pyplot as plt
import numpy as np

from synthraw.colour_space import resolve_colour_space
from synthraw.hdr import HDRReconstructionEngine

colour_resolution = resolve_colour_space(audit)
ProvenanceWriter().write(colour_resolution.records(), run.directory("metadata") / "colour_space.json")
hdr_engine = HDRReconstructionEngine()
scene_linear = hdr_engine.linearize(
    sr_rgb,
    run.directory("intermediates") / "03_scene_linear.npy",
    colour_space=colour_resolution,
    undo_tone=True,
)
# Stage separately because SR also publishes a source-support mask.
with tempfile.TemporaryDirectory(prefix="hdr_", dir=run.directory("intermediates")) as staging:
    hdr_result = hdr_engine.expand(
        sr_rgb,
        scene_linear,
        staging,
        highlight_ev=config.HDR_HIGHLIGHT_EV if config.HDR_ENABLED else 0.0,
        shadow_ev=config.HDR_SHADOW_EV if config.HDR_ENABLED else 0.0,
        backend=config.HDR_BACKEND if config.HDR_ENABLED else "DETERMINISTIC",
        negative_policy="CLIP",  # Explicit lossy gamut assumption, recorded in the audit.
    )
    for name, array in hdr_result.pop("arrays").items():
        source = Path(array.filename)
        destination = run.directory("intermediates" if name == "04_hdr_radiance" else "confidence") / source.name
        if name == "source_supported_mask" and destination.exists():
            destination.replace(destination.with_name("sr_source_supported_mask.npy"))
        source.replace(destination)
    del array
hdr_radiance = np.load(run.directory("intermediates") / "04_hdr_radiance.npy", mmap_mode="r")
(run.directory("metadata") / "hdr_reconstruction.json").write_text(
    json.dumps(hdr_result, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)

# Preview only a bounded sample; never materialize a full sensor-sized plotting array.
preview_step = max(1, int(np.ceil(max(sr_rgb.shape[:2]) / 256)))
preview = np.asarray(sr_rgb[::preview_step, ::preview_step], dtype=np.float32)
if sr_rgb.dtype == np.uint8:
    preview /= 255.0
fig, ax = plt.subplots(figsize=(6, 3))
for channel, colour in enumerate(("red", "green", "blue")):
    ax.hist(preview[..., channel].ravel(), bins=64, range=(0, 1), histtype="step", color=colour)
ax.set(title="SR encoded RGB histogram (sampled)", xlabel="Encoded value", ylabel="Sample count")
fig.tight_layout()
fig.savefig(run.directory("validation") / "input_histograms.png", dpi=100)
plt.close(fig)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
for ax, name in zip(axes, ("highlight_fabricated_mask", "shadow_fabricated_mask"), strict=True):
    mask = np.load(run.directory("confidence") / f"{name}.npy", mmap_mode="r")
    ax.imshow(mask[::preview_step, ::preview_step], vmin=0, vmax=1, cmap="gray")
    ax.set_title(name.replace("_fabricated_mask", " uncertainty"))
    ax.axis("off")
fig.tight_layout()
fig.savefig(run.directory("validation") / "clipping_masks.png", dpi=100)
plt.close(fig)
fig, ax = plt.subplots(figsize=(5, 3))
if sr_result.seams is not None:
    seam_map = np.load(sr_result.seams.residual_map_path, mmap_mode="r")
    seam_step = max(1, int(np.ceil(max(seam_map.shape) / 256)))
    plot = ax.imshow(seam_map[::seam_step, ::seam_step], cmap="magma", vmin=0)
    fig.colorbar(plot, ax=ax, label="Mean absolute RGB residual")
else:
    ax.text(0.5, 0.5, sr_result.seam_unavailable_reason, ha="center", va="center", wrap=True)
ax.set_title("SR tile seam diagnostic (sampled)")
ax.axis("off")
fig.tight_layout()
fig.savefig(run.directory("validation") / "tile_seam_map.png", dpi=100)
plt.close(fig)
print(f"Generated HDR area: {hdr_result['generated_hdr_area_percentage']:.2f}% (inferred/fabricated radiance)")
print(f"Gamut mapping audit: {hdr_result['gamut_mapping']}")

## Camera geometry + exposure inference

In [ ]:
import json

from synthraw.camera_geometry import CameraGeometryEstimator
from synthraw.camera_profile import CameraProfileManager
from synthraw.exposure import ExposurePosteriorEstimator
from synthraw.white_balance import WhiteBalanceEstimator

# Explicit hypothetical settings, never recovered capture facts.
MANUAL_WB = None  # Or {"manual_cct": 5500.0, "manual_tint": 0.0}.
MANUAL_EXPOSURE = None  # Required for MANUAL: (seconds, f-number, ISO).
REFERENCE_RAW = None  # User-supplied local file for REFERENCE_RAW mode.
SCENE_EV100 = 10.0  # Scene-scale prior, not brightness metering.
MANUAL_FOV_DEGREES = None

camera_profile = CameraProfileManager().build(config=config, geometry=geometry, reference_raw=REFERENCE_RAW)
camera_profile.write(run.directory("metadata"))
# Bounded sampling avoids copying the full HDR frame during inference.
camera_sample_step = max(1, int(np.ceil(max(hdr_radiance.shape[:2]) / 256)))
camera_sample = hdr_radiance[::camera_sample_step, ::camera_sample_step]
white_balance = WhiteBalanceEstimator().estimate(camera_sample, camera_profile, **(MANUAL_WB or {}))
exposure_posterior = ExposurePosteriorEstimator().estimate(
    camera_profile, mode=config.EXPOSURE_MODE, scene_ev100=SCENE_EV100, manual=MANUAL_EXPOSURE
)
exposure_records = exposure_posterior.records
camera_geometry = CameraGeometryEstimator().estimate(camera_profile, manual_fov_degrees=MANUAL_FOV_DEGREES)
for name, records in (
    ("white_balance", white_balance.records),
    ("exposure_posterior", exposure_records),
    ("camera_geometry", camera_geometry.records),
):
    ProvenanceWriter().write(list(records.values()), run.directory("metadata") / f"{name}.json")
print("Hypothetical exposure candidates (relative heuristic weights):")
print("Seconds       F-number   ISO       Weight")
for candidate in exposure_posterior.candidates:
    print(f"{candidate.exposure_time:<13.6g} {candidate.f_number:<10g} {candidate.iso:<9d} {candidate.weight:.6f}")

## Inverse ISP / camera colour

In [ ]:
from synthraw.camera_colour import CameraColourModel, InverseISPEngine

# The virtual identity colour fit is an explicitly fabricated prior.
colour_model = CameraColourModel()
inverse_isp = InverseISPEngine()
camera_native_rgb = np.lib.format.open_memmap(
    run.directory("intermediates") / "05_camera_native_rgb.npy",
    mode="w+",
    dtype=np.float32,
    shape=hdr_radiance.shape,
)
# HDR already undid tone processing. Preserve its radiance and headroom here.
CAMERA_STRIP_ROWS = 128
for start in range(0, hdr_radiance.shape[0], CAMERA_STRIP_ROWS):
    stop = min(start + CAMERA_STRIP_ROWS, hdr_radiance.shape[0])
    inverse_result = inverse_isp.reconstruct(
        hdr_radiance[start:stop],
        colour_model,
        white_balance_gains=white_balance.gains,
        use_raw_flow=config.USE_RAW_FLOW,
        use_spiraldiff=config.USE_SPIRALDIFF,
    )
    camera_native_rgb[start:stop] = inverse_result.camera_rgb
camera_native_rgb.flush()
inverse_isp_records = inverse_result.records
ProvenanceWriter().write(list(inverse_isp_records.values()), run.directory("metadata") / "inverse_isp.json")
print(f"Camera-native RGB saved; backend: {inverse_isp_records['inverse_isp_backend'].value}")

## Optical forward model

In [ ]:
from synthraw.optics import LensSimulator, OpticalBlurParameters

optical_sensor_rgb = np.lib.format.open_memmap(
    run.directory("intermediates") / "06_optical_sensor_rgb.npy",
    mode="w+",
    dtype=camera_native_rgb.dtype,
    shape=camera_native_rgb.shape,
)
optical_sensor_rgb, optical_records = LensSimulator().apply(
    camera_native_rgb,
    config,
    out=optical_sensor_rgb,
    strip_rows=CAMERA_STRIP_ROWS,
    work_dir=run.directory("intermediates"),
    blur=OpticalBlurParameters(
        lens_blur=True,
        f_number=exposure_records["FNumber"].value,
        pixel_pitch_um=camera_profile.camera["sensor_width_mm"].value * 1000 / geometry.width,
    ),
)
optical_sensor_rgb.flush()
ProvenanceWriter().write(
    [record for stage in optical_records.values() for record in stage.values()],
    run.directory("metadata") / "optical_parameters.json",
)
print(f"Optical sensor RGB saved; optical simulation enabled: {config.OPTICAL_SIMULATION}")

## Sensor/electron simulation

In [ ]:
# Sensor milestone placeholder:
# ElectronSignalSimulator and SensorNoiseSimulator: independent seeded sensor realisations.
print("Pending sensor milestone.")

## CFA construction

In [ ]:
# Sensor milestone placeholder:
# CFAMosaicer: Bayer CFA construction.
print("Pending sensor milestone.")

## ADC quantisation

In [ ]:
# Sensor milestone placeholder:
# ADCSimulator: 14-bit or explicitly virtual 16-bit quantisation.
print("Pending sensor milestone.")

## Metadata construction

In [ ]:
import json

from synthraw.provenance import ProvenanceWriter
from synthraw.sony_emulation import SonyARWEmulationBuilder
from synthraw.sony_registry import SonyMetadataRegistry


def construct_metadata(
    geometry, config, run, *, sensor_metadata, exposure=None, sony_inputs=None, optical_records=None
):
    """Use explicit sensor outputs; allocate no image arrays (PRD_v3 RAW milestone)."""
    from synthraw.dng_metadata import DNGMetadataBuilder

    required = {
        "cfa_phase",
        "black_level",
        "white_level",
        "color_matrix1",
        "color_matrix2",
        "as_shot_neutral",
        "noise_profile",
    }
    if required - sensor_metadata.keys():
        raise ValueError(f"Missing sensor metadata: {sorted(required - sensor_metadata.keys())}")
    records = DNGMetadataBuilder().build(geometry, config, exposure=exposure, **sensor_metadata)
    registry = SonyMetadataRegistry().build()
    emulation = SonyARWEmulationBuilder().build(
        registry,
        config,
        width=geometry.width,
        height=geometry.height,
        inputs=sony_inputs,
        optical_records=optical_records,
    )
    metadata_dir = run.directory("metadata")
    ProvenanceWriter().write(list(records.values()), metadata_dir / "dng_metadata.json")
    for name, payload in (("sony_tag_registry.json", registry), ("sony_arw_emulation.json", emulation)):
        (metadata_dir / name).write_text(json.dumps(payload, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return records


# The sensor milestone must supply the parameters used to generate the pixels.
# NoiseProfile must describe that same noise model. Matrices/WB remain labelled priors
# until a calibrated profile or estimator replaces them.
dng_metadata = None
if "sensor_metadata" in globals():
    dng_metadata = construct_metadata(
        geometry,
        config,
        run,
        sensor_metadata=globals()["sensor_metadata"],
        exposure=globals().get("exposure_records"),
        sony_inputs=globals().get("sony_metadata_inputs"),
        optical_records=globals().get("optical_records"),
    )
    print("DNG provenance and Sony sidecars written; RAW export still requires independent validation.")
else:
    print("Metadata construction ready; waiting for explicit sensor_metadata from the sensor milestone.")

## DNG export

In [ ]:
def export_dng(mosaic, records, config, run):
    """Publish only after structural and independent RAW validation (PRD_v3)."""
    from tempfile import TemporaryDirectory

    from synthraw.dng_writer import DNGWriter
    from synthraw.libraw_validator import LibRawValidator

    if not config.WRITE_DNG:
        return None
    if records is None:
        raise ValueError("Construct sensor metadata before DNG export")
    raw_dir = run.directory("raw")
    destination = raw_dir / "synthetic_sensor.dng"
    # Stage on the same filesystem so publication is atomic. Failed exports
    # remove partial files and leave any previously validated output intact.
    with TemporaryDirectory(prefix=".dng_export_", dir=raw_dir) as directory:
        from pathlib import Path

        staged = DNGWriter().write(Path(directory) / destination.name, mosaic, records, config)
        LibRawValidator().validate(staged, mosaic, records)
        staged.replace(destination)
    return destination


dng_path = None
if not config.WRITE_DNG:
    print("DNG export disabled by WRITE_DNG.")
elif "adc_mosaic" in globals() and dng_metadata is not None:
    dng_path = export_dng(globals()["adc_mosaic"], dng_metadata, config, run)
    print(f"DNG export passed structural and independent LibRaw validation: {dng_path}")
else:
    print("DNG export ready; waiting for adc_mosaic and constructed sensor metadata.")

## RAW round-trip development

In [ ]:
# Validation milestone placeholder:
# LibRawValidator and DemosaicValidator: independent RAW development.
print("Pending validation milestone.")

## 16-bit TIFF export

In [ ]:
# Integration milestone placeholder:
# TIFFWriter: 16-bit master export.
print("Pending integration milestone.")

## Validation suite

In [ ]:
# Validation milestone placeholder:
# NoiseValidator, ColourValidator and EditabilityValidator: scientific checks.
print("Pending validation milestone.")

## 2K/4K comparison

In [ ]:
# Validation milestone placeholder:
# SourceResolutionComparator: 2K/4K experiment.
print("Pending validation milestone.")

## Provenance + report packaging

In [ ]:
# Integration milestone placeholder:
# ProvenanceWriter and ReportGenerator: metadata records and report packaging.
print("Pending integration milestone.")